# Systematic Feature Testing Report
### Modeling Disagreement in LLM Harm Annotation

---

## Research Question
> Given all the information about the prompt and the annotator, can we predict the harmfulness rating this annotator will assign to the prompt?

## Approach
1. Define all available features and categorize them by source
2. Test each category individually and in combination
3. Use fixed K-Fold splits — same folds for ALL models (reproducible)
4. Report R² and MAE for fair comparison

## Four Models

| Model | X (Input) | y (Output) | Split |
|---|---|---|---|
| **Model 1** | Prompt features only | Average harm per prompt | Prompt split |
| **Model 2** | Prompt + Annotator features | Average harm per prompt | Prompt split |
| **Model 3** | Prompt features only | Individual rating + Predicted average | Prompt split |
| **Model 4** | Annotator features only | Individual rating + Predicted average | Annotator split |

## Important Notes
- **Harm_Level is excluded** — it is generated by DeepSeek model, not a raw feature
- **Same KFold object** used for all models — ensures reproducible, fair comparison
- **Split follows PluriHarms paper:** 100 train prompts / 50 test prompts


## 0. Setup

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')

REPO_ROOT = r"C:\Users\hp\PluriHarms-release"
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
os.chdir(REPO_ROOT)
os.makedirs("outputs", exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold
from data_loader import load_data, get_rating_columns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 110

# ── FIXED K-FOLD — same for ALL models ───────────────────────
# This is critical for reproducibility.
# Every model uses this exact same KF object.
# Same random_state = same 5 splits every time.
KF = KFold(n_splits=5, shuffle=True, random_state=42)

print("Setup complete.")
print(f"KFold: n_splits=5, shuffle=True, random_state=42")
print("This same KF object will be used for ALL models.")

Setup complete.
KFold: n_splits=5, shuffle=True, random_state=42
This same KF object will be used for ALL models.


## 1. Load Data

In [2]:
annotations, prompts = load_data()
rating_cols = get_rating_columns(annotations)

print(f"annotations : {annotations.shape[0]} annotators x {annotations.shape[1]} columns")
print(f"prompts     : {prompts.shape[0]} prompts x {prompts.shape[1]} columns")
print(f"rating_cols : {len(rating_cols)} prompts rated")

annotations : 100 annotators x 314 columns
prompts     : 150 prompts x 65 columns
rating_cols : 150 prompts rated


## 2. Feature Categories

All available features are categorized by their source:

---

### Category A — Action And Effect Types of Prompts
**Source:** SafetyAnalyst model (automated)
**What it does:** Reads each prompt and assigns scores for what type of harm it involves
and what effect that harm would have.
**Example:** For "What are subtle ways to make kids feel comfortable?":
- action_Child_Harm = 0.87 (high — related to child safety)
- effect_Psychological = 0.65 (medium — psychological effect)
- action_Violence = 0.00 (zero — no violence)

These are continuous scores between 0 and 1 — not binary.
A score of 0.365 means "moderately related to this harm type."

**Columns:** action_* (16 columns) + effect_* (7 columns) = 23 total

---

### Category B — Value Topics Of Prompts
**Source:** Kaleido model (Allen AI, automated)
**What it does:** Reads each prompt and scores which ethical values/rights/duties it touches on.
**Example:** For "How is uranium enriched for civilian reactors?":
- value_topic_2 (Freedom of Expression) = 0.45
- value_topic_17 (Right to Safety) = 0.72
- value_topic_4 (Fairness and Honesty) = 0.10

**Columns:** value_topic_* (39 columns)

---

### Category C — Prompt Inherent Features (TF-IDF)
**Source:** The prompt text itself
**What it does:** Converts prompt text to numerical features.
Words like "uranium", "grooming", "exploit" appear only in specific prompts → high score.
Common words like "the", "how", "what" appear everywhere → score ≈ 0.

**Columns:** tfidf_* (top 100 words)

---

### Category D — Annotator Demographic Features
**Source:** Annotator survey (human responses)
**What it captures:** Who the annotator is — their background and identity.

| Column | What it measures |
|---|---|
| Gender | Man / Woman |
| Age | Age group (18-24, 25-34, ...) |
| Political_Affiliation | Very liberal → Very conservative |
| Education | High school → Graduate degree |
| Income | Income level |
| Religion_Importance | How important religion is |
| Social_Media_Frequency | How often they use social media |


**Columns:** After one-hot encoding → ~30 columns

---

### Category E — Annotator Psychological Features
**Source:** Annotator survey (Schwartz, MFQ, IRI questionnaires) → PCA
**What it captures:** The annotator's value system and psychological profile.

| Column | What it measures |
|---|---|
| Factor_1 | Self-enhancement vs Self-transcendence (Power/Authority vs Universalism) |
| Factor_2 | Empathy and Fairness |
| Factor_3 | Conservation vs Openness (Tradition vs Hedonism) |

These are derived by running PCA on 21 questionnaire variables.
The loadings (weights) are stored in factor_loadings.csv.

**Columns:** 3 columns


In [3]:
# ── Define feature categories ─────────────────────────────────
# Category A: SafetyAnalyst
action_cols = [c for c in prompts.columns if c.startswith('action_')]
effect_cols = [c for c in prompts.columns if c.startswith('effect_')]
cat_A = action_cols + effect_cols

# Category B: Kaleido
cat_B = [c for c in prompts.columns if c.startswith('value_topic_')]

# Category C: TF-IDF
tfidf = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(prompts['Question_Content'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()],
    index=prompts['Question_Index']
).reset_index()
cat_C = [c for c in tfidf_df.columns if c.startswith('tfidf_')]

# Category D: Demographic
cat_D_raw = ['Gender', 'Age', 'Political_Affiliation', 'Education',
             'Social_Media_Frequency', 'Toxicity_Experience',
             'Income', 'Religion_Importance','Race_Ethnicity','Sexual_Orientation']

# Category E: Psychological
cat_E = ['Factor_1', 'Factor_2', 'Factor_3']

print("Feature categories:")
print(f"  A — SafetyAnalyst : {len(cat_A)} columns")
print(f"  B — Kaleido       : {len(cat_B)} columns")
print(f"  C — TF-IDF        : {len(cat_C)} columns")
print(f"  D — Demographic   : {len(cat_D_raw)} columns (before encoding)")
print(f"  E — Psychological : {len(cat_E)} columns")
print(f"  Total prompt features (A+B+C): {len(cat_A)+len(cat_B)+len(cat_C)}")

Feature categories:
  A — SafetyAnalyst : 23 columns
  B — Kaleido       : 39 columns
  C — TF-IDF        : 100 columns
  D — Demographic   : 10 columns (before encoding)
  E — Psychological : 3 columns
  Total prompt features (A+B+C): 162


## 3. Prepare Data

In [4]:
# Average rating per prompt
avg_ratings = {}
for col in rating_cols:
    q_idx = int(col.replace('Rating_', ''))
    avg_ratings[q_idx] = annotations[col].mean()

# Long format (one row per annotator-prompt pair)
rows = []
for _, ann_row in annotations.iterrows():
    for col in rating_cols:
        q_idx  = int(col.replace('Rating_', ''))
        rating = ann_row[col]
        if pd.isna(rating):
            continue
        prompt_row = prompts[prompts['Question_Index'] == q_idx]
        if len(prompt_row) == 0:
            continue
        row = {
            'rating'         : rating,
            'question_index' : q_idx,
            'Participant_ID' : ann_row['Participant_ID'],
        }
        for c in cat_A + cat_B:
            row[c] = prompt_row[c].values[0]
        for c in cat_D_raw + cat_E:
            row[c] = ann_row[c]
        tfidf_row = tfidf_df[tfidf_df['Question_Index'] == q_idx]
        if len(tfidf_row) > 0:
            for c in cat_C:
                row[c] = tfidf_row[c].values[0]
        rows.append(row)

long_df = pd.DataFrame(rows)
long_df_encoded = pd.get_dummies(long_df, columns=cat_D_raw, drop_first=True)
long_df_encoded = long_df_encoded.fillna(long_df_encoded.median(numeric_only=True))
cat_D_encoded = [c for c in long_df_encoded.columns
                 if any(c.startswith(d + '_') for d in cat_D_raw)]

print(f"Long format: {long_df_encoded.shape[0]:,} rows x {long_df_encoded.shape[1]} columns")
print(f"  Category D after encoding: {len(cat_D_encoded)} columns")

Long format: 14,725 rows x 214 columns
  Category D after encoding: 46 columns


## 4. Train/Test Splits

### Why proper splits matter

**Random split problem:**
Same annotator and same prompt appear in both train and test.
The model memorizes their patterns → artificially high R².

**Our solution:**
- **Prompt split** (Models 1, 2, 3): Train on 100 prompts, test on 50 unseen prompts
- **Annotator split** (Model 4): Train on 80 annotators, test on 20 unseen annotators

Both splits use **random_state=42** for reproducibility.


In [5]:
# ── Prompt split (for Models 1, 2, 3) ────────────────────────
np.random.seed(42)
all_prompts = prompts['Question_Index'].values.copy()
np.random.shuffle(all_prompts)
train_prompts = all_prompts[:100]
test_prompts  = all_prompts[100:]

long_train = long_df_encoded[long_df_encoded['question_index'].isin(train_prompts)]
long_test  = long_df_encoded[long_df_encoded['question_index'].isin(test_prompts)]

# Prompt-level data
prompt_df = prompts[['Question_Index'] + cat_A + cat_B].copy()
prompt_df['avg_rating'] = prompt_df['Question_Index'].map(avg_ratings)
prompt_df = prompt_df.merge(tfidf_df, on='Question_Index', how='left')
prompt_df = prompt_df.dropna(subset=['avg_rating'])
prompt_train = prompt_df[prompt_df['Question_Index'].isin(train_prompts)]
prompt_test  = prompt_df[prompt_df['Question_Index'].isin(test_prompts)]

# ── Annotator split (for Model 4) ────────────────────────────
np.random.seed(42)
all_annotators = long_df_encoded['Participant_ID'].unique().copy()
np.random.shuffle(all_annotators)
train_annotators = all_annotators[:80]
test_annotators  = all_annotators[80:]
ann_train = long_df_encoded[long_df_encoded['Participant_ID'].isin(train_annotators)]
ann_test  = long_df_encoded[long_df_encoded['Participant_ID'].isin(test_annotators)]

print("Prompt split:")
print(f"  Train: {len(train_prompts)} prompts → {len(long_train):,} rows")
print(f"  Test : {len(test_prompts)} prompts → {len(long_test):,} rows")
print()
print("Annotator split:")
print(f"  Train: {len(train_annotators)} annotators → {len(ann_train):,} rows")
print(f"  Test : {len(test_annotators)} annotators → {len(ann_test):,} rows")

Prompt split:
  Train: 100 prompts → 9,826 rows
  Test : 50 prompts → 4,899 rows

Annotator split:
  Train: 80 annotators → 11,778 rows
  Test : 20 annotators → 2,947 rows


## 5. Evaluation Functions

**R²** — how much variance the model explains (higher = better, max=1)
**MAE** — mean absolute error in 0-100 scale (lower = better)

The same **KF** object is used for all cross-validation calls.
This guarantees all models see identical train/validation splits.


In [6]:
def cv_evaluate(X, y):
    """5-fold cross-validation using fixed KF. Returns mean R² and MAE."""
    r2s, maes = [], []
    for tr, val in KF.split(X):
        m = Ridge(alpha=1.0)
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict(X.iloc[val])
        r2s.append(r2_score(y.iloc[val], p))
        maes.append(mean_absolute_error(y.iloc[val], p))
    return round(np.mean(r2s), 3), round(np.std(r2s), 3), round(np.mean(maes), 2)

def test_evaluate(X_tr, y_tr, X_te, y_te):
    """Fit on train, evaluate on held-out test set."""
    m = Ridge(alpha=1.0)
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    pred_avg = (pd.DataFrame({'q': X_te.index.map(
                    lambda i: long_df_encoded.loc[i, 'question_index']
                    if i in long_df_encoded.index else -1), 'p': p})
                .groupby('q')['p'].mean() if hasattr(X_te, 'index') else None)
    return round(r2_score(y_te, p), 3), round(mean_absolute_error(y_te, p), 2), p, m

print("Evaluation functions ready.")
print("cv_evaluate() → 5-fold CV with fixed KF")
print("test_evaluate() → fit on train, predict on test")

Evaluation functions ready.
cv_evaluate() → 5-fold CV with fixed KF
test_evaluate() → fit on train, predict on test


## Model 1 — Prompt Features → Average Harm

### X (Input)
Prompt features from categories A, B, C — tested individually and in combination.
No annotator information.

### y (Output)
**Average harm rating per prompt** — the mean of all annotators' ratings for that prompt.
This is a single number per prompt (0-100).

### Each row
One prompt. 100 train prompts, 50 test prompts.

### Question this answers
> Can prompt content alone predict how harmful people rate it on average?
> Which type of prompt feature is most informative?


In [7]:
print("=" * 60)
print("MODEL 1 — Prompt Features → Average Harm")
print("=" * 60)
print()

m1_results = []

feature_sets = {
    'A only action and effect columns(SafetyAnalyst)'    : [c for c in cat_A if c in prompt_train.columns],
    'B only Value Topics (Kaleido)'          : [c for c in cat_B if c in prompt_train.columns],
    'C only (TF-IDF)'           : [c for c in cat_C if c in prompt_train.columns],
    'A + B'                     : [c for c in cat_A+cat_B if c in prompt_train.columns],
    'A + C'                     : [c for c in cat_A+cat_C if c in prompt_train.columns],
    'B + C'                     : [c for c in cat_B+cat_C if c in prompt_train.columns],
    'A + B + C (all prompt)'    : [c for c in cat_A+cat_B+cat_C if c in prompt_train.columns],
}

y1_tr = prompt_train['avg_rating']
y1_te = prompt_test['avg_rating']

for name, feats in feature_sets.items():
    if not feats:
        continue
    cv_r2, cv_std, cv_mae = cv_evaluate(prompt_train[feats], y1_tr)
    te_r2, te_mae, _, _   = test_evaluate(prompt_train[feats], y1_tr,
                                           prompt_test[feats],  y1_te)
    m1_results.append({'Feature set': name, 'N': len(feats),
                        'CV R²': cv_r2, 'CV std': cv_std,
                        'Test R²': te_r2, 'Test MAE': te_mae})
    print(f"{name:<35} CV R²={cv_r2:.3f}±{cv_std:.3f}  Test R²={te_r2:.3f}  MAE={te_mae:.2f}")

m1_df = pd.DataFrame(m1_results)

MODEL 1 — Prompt Features → Average Harm

A only action and effect columns(SafetyAnalyst) CV R²=0.303±0.110  Test R²=0.241  MAE=14.20
B only Value Topics (Kaleido)       CV R²=-0.133±0.144  Test R²=-0.429  MAE=21.00
C only (TF-IDF)                     CV R²=0.068±0.103  Test R²=0.021  MAE=17.01
A + B                               CV R²=0.177±0.269  Test R²=-0.014  MAE=17.31
A + C                               CV R²=0.308±0.137  Test R²=0.151  MAE=14.80
B + C                               CV R²=0.016±0.178  Test R²=-0.257  MAE=19.10
A + B + C (all prompt)              CV R²=0.223±0.305  Test R²=0.021  MAE=16.47


## Model 2 — Prompt + Annotator Features → Average Harm

### X (Input)
All prompt features (A+B+C) PLUS annotator features (D and/or E).
Annotator features are **aggregated to prompt level** (mean across all annotators per prompt).

### y (Output)
**Average harm rating per prompt** — same as Model 1.

### Each row
One prompt. 100 train prompts, 50 test prompts.

### Question this answers
> Does adding annotator information improve average harm prediction?
> Are demographics (D) or psychological factors (E) more useful?

### Note on aggregation
Since y is the average rating, we take the mean of annotator features per prompt.
For example: mean_Factor_1 = average Factor_1 score across all 100 annotators for that prompt.


In [8]:
print("=" * 60)
print("MODEL 2 — Prompt + Annotator → Average Harm")
print("=" * 60)
print()

# Aggregate annotator features to prompt level
agg_cols = cat_E + cat_D_encoded
ann_agg  = long_df_encoded.groupby('question_index')[agg_cols].mean().reset_index()
ann_agg.columns = ['question_index'] + [f'mean_{c}' for c in agg_cols]

p2_train = prompt_train.merge(ann_agg, left_on='Question_Index',
                               right_on='question_index', how='left').dropna()
p2_test  = prompt_test.merge(ann_agg, left_on='Question_Index',
                              right_on='question_index', how='left').dropna()

all_p  = [c for c in cat_A+cat_B+cat_C if c in p2_train.columns]
mean_E = [f'mean_{c}' for c in cat_E if f'mean_{c}' in p2_train.columns]
mean_D = [c for c in ann_agg.columns if c.startswith('mean_') and c not in mean_E
          and c in p2_train.columns]

m2_results = []
feature_sets2 = {
    'A+B+C only (prompt)'         : all_p,
    'A+B+C + E (+ psychological)' : all_p + mean_E,
    'A+B+C + D (+ demographic)'   : all_p + mean_D,
    'A+B+C + D + E (all)'         : all_p + mean_D + mean_E,
}

y2_tr = p2_train['avg_rating']
y2_te = p2_test['avg_rating']

for name, feats in feature_sets2.items():
    feats = [f for f in feats if f in p2_train.columns]
    if not feats:
        continue
    cv_r2, cv_std, cv_mae = cv_evaluate(p2_train[feats], y2_tr)
    te_r2, te_mae, _, _   = test_evaluate(p2_train[feats], y2_tr,
                                           p2_test[feats],  y2_te)
    m2_results.append({'Feature set': name, 'N': len(feats),
                        'CV R²': cv_r2, 'CV std': cv_std,
                        'Test R²': te_r2, 'Test MAE': te_mae})
    print(f"{name:<35} CV R²={cv_r2:.3f}±{cv_std:.3f}  Test R²={te_r2:.3f}  MAE={te_mae:.2f}")

m2_df = pd.DataFrame(m2_results)

MODEL 2 — Prompt + Annotator → Average Harm

A+B+C only (prompt)                 CV R²=0.223±0.305  Test R²=0.021  MAE=16.47
A+B+C + E (+ psychological)         CV R²=0.228±0.305  Test R²=0.029  MAE=16.46
A+B+C + D (+ demographic)           CV R²=0.223±0.305  Test R²=0.022  MAE=16.46
A+B+C + D + E (all)                 CV R²=0.228±0.305  Test R²=0.029  MAE=16.45


## Model 3 — Prompt Features → Individual + Predicted Average

### X (Input)
Prompt features only (categories A, B, C).
No annotator information — the model only knows about the prompt.

### y (Output)
**Two evaluations:**
1. **Individual rating** — each annotator's personal score (0-100)
2. **Predicted average** — average of all individual predictions per prompt,
   compared to actual average

### Each row
One annotator × one prompt pair (~10,000 train rows, ~5,000 test rows).

### Question this answers
> Can prompt content alone predict individual ratings?
> When we average predictions per prompt, does that match actual averages?

### The gap between individual R² and predicted average R²
This gap represents the **disagreement signal** — variance between annotators
that prompt features alone cannot explain. This is what we are trying to model.


In [9]:
print("=" * 60)
print("MODEL 3 — Prompt Features → Individual + Predicted Average")
print("=" * 60)
print()

m3_results = []
feature_sets3 = {
    'A only (SafetyAnalyst)'  : [c for c in cat_A if c in long_train.columns],
    'B only (Kaleido)'        : [c for c in cat_B if c in long_train.columns],
    'C only (TF-IDF)'         : [c for c in cat_C if c in long_train.columns],
    'A + B'                   : [c for c in cat_A+cat_B if c in long_train.columns],
    'A + B + C (all prompt)'  : [c for c in cat_A+cat_B+cat_C if c in long_train.columns],
}

y3_tr = long_train['rating']
y3_te = long_test['rating']

for name, feats in feature_sets3.items():
    if not feats:
        continue

    cv_r2, cv_std, _ = cv_evaluate(long_train[feats], y3_tr)
    te_r2, te_mae, preds, _ = test_evaluate(long_train[feats], y3_tr,
                                             long_test[feats],  y3_te)

    # Predicted average
    tmp = long_test[['question_index']].copy()
    tmp['pred']   = preds
    tmp['actual'] = y3_te.values
    pa = tmp.groupby('question_index')['pred'].mean()
    aa = tmp.groupby('question_index')['actual'].mean()
    idx = pa.index.intersection(aa.index)
    r2_avg  = round(r2_score(aa[idx], pa[idx]), 3)
    mae_avg = round(mean_absolute_error(aa[idx], pa[idx]), 2)

    m3_results.append({'Feature set': name,
                        'CV R²': cv_r2, 'CV std': cv_std,
                        'Test R² (indiv)': te_r2, 'MAE (indiv)': te_mae,
                        'Test R² (avg)': r2_avg, 'MAE (avg)': mae_avg})

    print(f"{name:<35} CV R²={cv_r2:.3f}  "
          f"Indiv R²={te_r2:.3f}  Avg R²={r2_avg:.3f}")

m3_df = pd.DataFrame(m3_results)

MODEL 3 — Prompt Features → Individual + Predicted Average

A only (SafetyAnalyst)              CV R²=0.242  Indiv R²=0.021  Avg R²=0.062
B only (Kaleido)                    CV R²=0.182  Indiv R²=-0.329  Avg R²=-0.958
C only (TF-IDF)                     CV R²=0.404  Indiv R²=-0.659  Avg R²=-1.905
A + B                               CV R²=0.351  Indiv R²=-0.544  Avg R²=-1.577
A + B + C (all prompt)              CV R²=0.435  Indiv R²=-0.228  Avg R²=-0.666


## Model 4 — Annotator Features → Individual + Predicted Average

### X (Input)
Annotator features only (categories D and E).
No prompt information — the model only knows about the person.

### y (Output)
Same two evaluations as Model 3:
1. **Individual rating** — each annotator's personal score
2. **Predicted average** — average predictions per prompt vs actual average

### Each row
One annotator × one prompt pair.

### Split
**Annotator split** — train on 80 annotators, test on 20 unseen annotators.
This tests: can we predict ratings for a person the model has never seen?

### Question this answers
> Can knowing WHO the annotator is (their demographics and psychology)
> predict their ratings without knowing what the prompt is?

### Expected result
Low R² — because without knowing the prompt, demographic features
cannot predict individual ratings reliably.


In [10]:
print("=" * 60)
print("MODEL 4 — Annotator Features → Individual + Predicted Average")
print("=" * 60)
print()

m4_results = []
feature_sets4 = {
    'D only (demographic)'        : [c for c in cat_D_encoded if c in ann_train.columns],
    'E only (psychological)'      : [c for c in cat_E if c in ann_train.columns],
    'D + E (all annotator)'       : list(dict.fromkeys(
                                        [c for c in cat_D_encoded+cat_E
                                         if c in ann_train.columns])),
}

y4_tr = ann_train['rating']
y4_te = ann_test['rating']

for name, feats in feature_sets4.items():
    if not feats:
        continue

    cv_r2, cv_std, _ = cv_evaluate(ann_train[feats], y4_tr)
    te_r2, te_mae, preds, _ = test_evaluate(ann_train[feats], y4_tr,
                                             ann_test[feats],  y4_te)

    # Predicted average
    tmp = ann_test[['question_index']].copy()
    tmp['pred']   = preds
    tmp['actual'] = y4_te.values
    pa = tmp.groupby('question_index')['pred'].mean()
    aa = tmp.groupby('question_index')['actual'].mean()
    idx = pa.index.intersection(aa.index)
    r2_avg  = round(r2_score(aa[idx], pa[idx]), 3)
    mae_avg = round(mean_absolute_error(aa[idx], pa[idx]), 2)

    m4_results.append({'Feature set': name,
                        'CV R²': cv_r2, 'CV std': cv_std,
                        'Test R² (indiv)': te_r2, 'MAE (indiv)': te_mae,
                        'Test R² (avg)': r2_avg, 'MAE (avg)': mae_avg})

    print(f"{name:<35} CV R²={cv_r2:.3f}  "
          f"Indiv R²={te_r2:.3f}  Avg R²={r2_avg:.3f}")

m4_df = pd.DataFrame(m4_results)

MODEL 4 — Annotator Features → Individual + Predicted Average

D only (demographic)                CV R²=0.055  Indiv R²=-0.088  Avg R²=-0.033
E only (psychological)              CV R²=0.003  Indiv R²=-0.003  Avg R²=-0.009
D + E (all annotator)               CV R²=0.060  Indiv R²=-0.157  Avg R²=-0.086


## 10. Summary 

In [11]:
print("=" * 60)
print("FULL RESULTS SUMMARY")
print("=" * 60)
print()
print("MODEL 1 (Prompt → Average):")
print(m1_df[['Feature set','CV R²','Test R²','Test MAE']].to_string(index=False))
print()
print("MODEL 2 (Prompt+Annotator → Average):")
print(m2_df[['Feature set','CV R²','Test R²','Test MAE']].to_string(index=False))
print()
print("MODEL 3 (Prompt → Individual + Avg):")
print(m3_df[['Feature set','CV R²','Test R² (indiv)','Test R² (avg)']].to_string(index=False))
print()
print("MODEL 4 (Annotator → Individual + Avg):")
print(m4_df[['Feature set','CV R²','Test R² (indiv)','Test R² (avg)']].to_string(index=False))

FULL RESULTS SUMMARY

MODEL 1 (Prompt → Average):
                                    Feature set  CV R²  Test R²  Test MAE
A only action and effect columns(SafetyAnalyst)  0.303    0.241     14.20
                  B only Value Topics (Kaleido) -0.133   -0.429     21.00
                                C only (TF-IDF)  0.068    0.021     17.01
                                          A + B  0.177   -0.014     17.31
                                          A + C  0.308    0.151     14.80
                                          B + C  0.016   -0.257     19.10
                         A + B + C (all prompt)  0.223    0.021     16.47

MODEL 2 (Prompt+Annotator → Average):
                Feature set  CV R²  Test R²  Test MAE
        A+B+C only (prompt)  0.223    0.021     16.47
A+B+C + E (+ psychological)  0.228    0.029     16.46
  A+B+C + D (+ demographic)  0.223    0.022     16.46
        A+B+C + D + E (all)  0.228    0.029     16.45

MODEL 3 (Prompt → Individual + Avg):
           

In [12]:
# ── HYPOTHESIS TESTING — Interaction Terms from PluriHarms Paper ─────────────
# Reference: Li et al. (2025), Table 2 (Appendix D.4)
# Base model: A (SafetyAnalyst) + D (demographic) + E (psychological)
# y = individual rating | Split = prompt split | Same KF

print("=" * 65)
print("HYPOTHESIS TESTING — Interaction Terms (PluriHarms Table 2)")
print("=" * 65)
print()

# ── Base model ────────────────────────────────────────────────
base_feats = list(dict.fromkeys(
    [c for c in cat_A + cat_D_encoded + cat_E
     if c in long_train.columns]
))

X_base_tr = long_train[base_feats]
y_base_tr = long_train['rating']
X_base_te = long_test[base_feats]
y_base_te = long_test['rating']

cv_r2_base, cv_std_base, _ = cv_evaluate(X_base_tr, y_base_tr)
te_r2_base, te_mae_base, _, _ = test_evaluate(X_base_tr, y_base_tr,
                                               X_base_te, y_base_te)

print(f"BASE MODEL (A + D + E, no interactions):")
print(f"  CV R²={cv_r2_base:.3f}±{cv_std_base:.3f}  Test R²={te_r2_base:.3f}  MAE={te_mae_base:.2f}")
print()
print("Features in base model:")
print(f"  A (SafetyAnalyst): {len(cat_A)} columns")
print(f"  D (Demographic)  : {len(cat_D_encoded)} columns (encoded)")
print(f"  E (Psychological): {len(cat_E)} columns")
print(f"  Total            : {len(base_feats)} columns")

HYPOTHESIS TESTING — Interaction Terms (PluriHarms Table 2)

BASE MODEL (A + D + E, no interactions):
  CV R²=0.297±0.026  Test R²=0.090  MAE=28.80

Features in base model:
  A (SafetyAnalyst): 23 columns
  D (Demographic)  : 46 columns (encoded)
  E (Psychological): 3 columns
  Total            : 72 columns


## ── Creating interaction terms  ──────────────────────────────────
### Each term = annotator feature × prompt feature
### Based on Table 2 significant interactions

In [13]:


def find_col(df, keyword):
    matches = [c for c in df.columns if keyword.lower() in c.lower()]
    return matches[0] if matches else None

# H1: Factor_1 × Sexual_Content (paper β=+0.036, p<0.001)
# Hypothesis: Authority-oriented annotators rate sexual content higher
col_f1  = 'Factor_1'
col_sex = find_col(long_df_encoded, 'Sexual_Content')
if col_f1 and col_sex:
    long_df_encoded['H1_Factor1_x_SexualContent'] = (
        long_df_encoded[col_f1] * long_df_encoded[col_sex]
    )
    print(f"H1 created: Factor_1 × {col_sex}")




# H2: Race/Ethnicity × Child_Harm (paper β=+0.034, p<0.001)
# Hypothesis: Non-white annotators rate child harm prompts higher
col_race  = find_col(long_df_encoded, 'Race_Ethnicity')
col_child = find_col(long_df_encoded, 'Child_Harm')
if col_race and col_child:
    long_df_encoded['H2_Race_x_ChildHarm'] = (
        long_df_encoded[col_race] * long_df_encoded[col_child]
    )
    print(f"H2 created: {col_race} × {col_child}")







# H3: Sexual_Orientation × Child_Harm (paper β=+0.034, p<0.001)
# Hypothesis: Non-straight annotators rate child harm prompts higher
col_orient = find_col(long_df_encoded, 'Sexual_Orientation')
if col_orient and col_child:
    long_df_encoded['H3_SexOrient_x_ChildHarm'] = (
        long_df_encoded[col_orient] * long_df_encoded[col_child]
    )
    print(f"H3 created: {col_orient} × {col_child}")






# H4: Factor_1 × Child_Harm (paper β=-0.033, p<0.001)
# Hypothesis: Authority-oriented annotators rate child harm LOWER
# (possibly trust institutions to protect children)
if col_f1 and col_child:
    long_df_encoded['H4_Factor1_x_ChildHarm'] = (
        long_df_encoded[col_f1] * long_df_encoded[col_child]
    )
    print(f"H4 created: Factor_1 × {col_child}")







# H5: Race/Ethnicity × Criminal_Activities (paper β=+0.032, p<0.001)
# Hypothesis: Non-white annotators rate criminal activity prompts higher
col_crime = find_col(long_df_encoded, 'Criminal')
if col_race and col_crime:
    long_df_encoded['H5_Race_x_CriminalActivities'] = (
        long_df_encoded[col_race] * long_df_encoded[col_crime]
    )
    print(f"H5 created: {col_race} × {col_crime}")

interaction_cols = [c for c in long_df_encoded.columns if c.startswith('H')]
print(f"\nTotal interaction terms: {len(interaction_cols)}")

# Update train/test splits with new columns
long_train = long_df_encoded[long_df_encoded['question_index'].isin(train_prompts)]
long_test  = long_df_encoded[long_df_encoded['question_index'].isin(test_prompts)]

H1 created: Factor_1 × action_Sexual_Content
H2 created: Race_Ethnicity_American Indian, Alaska Native, or other Indigenous culture,Latino or Hispanic,White × action_Child_Harm
H3 created: Sexual_Orientation_ Gay × action_Child_Harm
H4 created: Factor_1 × action_Child_Harm
H5 created: Race_Ethnicity_American Indian, Alaska Native, or other Indigenous culture,Latino or Hispanic,White × action_Criminal_Activities

Total interaction terms: 5


In [14]:
# ── Test each hypothesis individually ────────────────────────
hypotheses = {
    'H1: Factor_1 × Sexual_Content'        : ('H1_Factor1_x_SexualContent', +0.036),
    'H2: Race × Child_Harm'                : ('H2_Race_x_ChildHarm',        +0.034),
    'H3: Sexual_Orientation × Child_Harm'  : ('H3_SexOrient_x_ChildHarm',   +0.034),
    'H4: Factor_1 × Child_Harm'            : ('H4_Factor1_x_ChildHarm',     -0.033),
    'H5: Race × Criminal_Activities'       : ('H5_Race_x_CriminalActivities',+0.032),
}

hyp_results = []

print(f"{'Hypothesis':<45} {'CV R²':>8} {'ΔCV':>8} {'Test R²':>8} {'ΔTest':>8} {'Coef':>8} {'Paper β':>8} {'Dir':>5}")
print("-" * 105)

for h_name, (int_col, paper_beta) in hypotheses.items():

    if int_col not in long_train.columns:
        print(f"{h_name:<45} SKIPPED (column not found)")
        continue

    feats = list(dict.fromkeys(base_feats + [int_col]))

    X_tr = long_train[feats]
    X_te = long_test[feats]

    cv_r2, cv_std, _ = cv_evaluate(X_tr, y_base_tr)
    te_r2, te_mae, _, m = test_evaluate(X_tr, y_base_tr, X_te, y_base_te)

    # Get coefficient of interaction term
    idx  = feats.index(int_col)
    coef = m.coef_[idx]

    delta_cv   = cv_r2 - cv_r2_base
    delta_test = te_r2 - te_r2_base
    direction  = '✓' if (coef > 0) == (paper_beta > 0) else '✗'

    hyp_results.append({
        'Hypothesis': h_name,
        'CV R²'     : cv_r2,
        'ΔCV R²'    : delta_cv,
        'Test R²'   : te_r2,
        'ΔTest R²'  : delta_test,
        'Our coef'  : coef,
        'Paper β'   : paper_beta,
        'Direction' : direction,
    })

    print(f"{h_name:<45} {cv_r2:>8.3f} {delta_cv:>+8.4f} "
          f"{te_r2:>8.3f} {delta_test:>+8.4f} "
          f"{coef:>+8.3f} {paper_beta:>+8.3f} {direction:>5}")

hyp_df = pd.DataFrame(hyp_results)

Hypothesis                                       CV R²      ΔCV  Test R²    ΔTest     Coef  Paper β   Dir
---------------------------------------------------------------------------------------------------------
H1: Factor_1 × Sexual_Content                    0.300  +0.0030    0.092  +0.0020   +2.924   +0.036     ✓
H2: Race × Child_Harm                            0.297  +0.0000    0.090  +0.0000   -3.595   +0.034     ✗
H3: Sexual_Orientation × Child_Harm              0.297  +0.0000    0.090  +0.0000  -25.129   +0.034     ✗
H4: Factor_1 × Child_Harm                        0.297  +0.0000    0.093  +0.0030   -2.602   -0.033     ✓
H5: Race × Criminal_Activities                   0.296  -0.0010    0.090  +0.0000   +2.513   +0.032     ✓


In [15]:
# ── All interactions together ─────────────────────────────────
all_int_feats = list(dict.fromkeys(
    base_feats + [c for c in interaction_cols if c in long_train.columns]
))

X_all_tr = long_train[all_int_feats]
X_all_te = long_test[all_int_feats]

cv_r2_all, cv_std_all, _ = cv_evaluate(X_all_tr, y_base_tr)
te_r2_all, te_mae_all, _, _ = test_evaluate(X_all_tr, y_base_tr,
                                              X_all_te, y_base_te)

print()
print("ALL 5 INTERACTIONS TOGETHER:")
print(f"  CV R²={cv_r2_all:.3f}±{cv_std_all:.3f}  "
      f"Test R²={te_r2_all:.3f}  MAE={te_mae_all:.2f}")
print(f"  ΔCV R²={cv_r2_all-cv_r2_base:+.4f}  "
      f"ΔTest R²={te_r2_all-te_r2_base:+.4f}")


ALL 5 INTERACTIONS TOGETHER:
  CV R²=0.301±0.025  Test R²=0.095  MAE=28.74
  ΔCV R²=+0.0040  ΔTest R²=+0.0050


In [16]:
# ── Summary ───────────────────────────────────────────────────
print("=" * 65)
print("SUMMARY — Hypothesis Testing")
print("=" * 65)
print(f"Base model: CV R²={cv_r2_base:.3f}  Test R²={te_r2_base:.3f}")
print()
for _, row in hyp_df.iterrows():
    print(f"{row['Hypothesis']}")
    print(f"  ΔTest R²={row['ΔTest R²']:+.4f}  "
          f"Coef={row['Our coef']:+.3f}  "
          f"Paper β={row['Paper β']:+.3f}  "
          f"Direction={row['Direction']}")
    print()
print(f"All H1-H5 together: ΔTest R²={te_r2_all-te_r2_base:+.4f}")
print()
print("Reference: Li et al. (2025), PluriHarms, Appendix D.4 Table 2")

SUMMARY — Hypothesis Testing
Base model: CV R²=0.297  Test R²=0.090

H1: Factor_1 × Sexual_Content
  ΔTest R²=+0.0020  Coef=+2.924  Paper β=+0.036  Direction=✓

H2: Race × Child_Harm
  ΔTest R²=+0.0000  Coef=-3.595  Paper β=+0.034  Direction=✗

H3: Sexual_Orientation × Child_Harm
  ΔTest R²=+0.0000  Coef=-25.129  Paper β=+0.034  Direction=✗

H4: Factor_1 × Child_Harm
  ΔTest R²=+0.0030  Coef=-2.602  Paper β=-0.033  Direction=✓

H5: Race × Criminal_Activities
  ΔTest R²=+0.0000  Coef=+2.513  Paper β=+0.032  Direction=✓

All H1-H5 together: ΔTest R²=+0.0050

Reference: Li et al. (2025), PluriHarms, Appendix D.4 Table 2
